# MNIST Machine Learning Pipeline with CV Filters
This notebook demonstrates a traditional machine learning pipeline that:
1. Loads the MNIST dataset.
2. Applies Computer Vision filters (Gaussian Blur + Sobel Edge Detection).
3. Standardizes and prepares the data.
4. Trains a Classifier (Logistic Regression or k-NN).
5. Evaluates performance using metrics and a confusion matrix.

## Step 1: Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import time

## Step 2: Load the MNIST Dataset
We will fetch the MNIST dataset from OpenML and split it into training and testing sets.

In [ ]:
# This might take a minute to download the first time
print("Downloading MNIST dataset...")
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
print(f"Dataset downloaded. Shape: {X.shape}")

# Reshape 1D vectors back into 2D (28x28) images for computer vision filtering
X_images = X.reshape(-1, 28, 28).astype('uint8')

# Split into Train (60,000) and Test (10,000) to match standard MNIST
X_train_img, X_test_img, y_train, y_test = train_test_split(
    X_images, y, test_size=10000, random_state=42, stratify=y
)
print(f"Train shape: {X_train_img.shape}, Test shape: {X_test_img.shape}")

## Step 3: Define and Apply CV Filters
We apply a Gaussian Blur to smooth out noise followed by a Sobel filter to extract structural edge data.

In [ ]:
def apply_cv_filters(images):
    processed_images = []
    for img in images:
        # 1. Apply Gaussian Blur to smooth minor handwriting artifacts
        blurred = cv2.GaussianBlur(img, (3, 3), 0)
        
        # 2. Extract Horizontal and Vertical Edges using Sobel filters
        sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
        sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
        
        # 3. Calculate absolute magnitude of gradients
        magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
        
        # 4. Normalize magnitude map back to 0-255 range
        magnitude = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX).astype('uint8')
        
        # 5. Flatten the 28x28 processed matrix into a 784-dimensional vector
        processed_images.append(magnitude.flatten())
        
    # Normalize all pixel intensities to a stable 0.0 - 1.0 range
    return np.array(processed_images) / 255.0

print("Applying CV filters to training data...")
X_train_proc = apply_cv_filters(X_train_img)

print("Applying CV filters to testing data...")
X_test_proc = apply_cv_filters(X_test_img)
print("Processing complete!")

### Visualize the Filtering Effect
Let's visualize a raw image versus its edge-filtered representation.

In [ ]:
plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1)
plt.imshow(X_train_img[0], cmap='gray')
plt.title(f"Raw Image (Label: {y_train[0]})")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(X_train_proc[0].reshape(28, 28), cmap='gray')
plt.title("Filtered Edge Image")
plt.axis('off')
plt.show()

## Step 4: Train a Classifier
Toggle between `Logistic Regression` and `k-NN` to see how they perform. 
*Note: k-NN inference can be slow on large datasets, so we use a small subset of the test data for quick execution during step 5 if k-NN is selected.*

In [ ]:
# CHOOSE YOUR MODEL HERE: 'logistic_regression' or 'knn'
MODEL_CHOICE = 'logistic_regression' 

if MODEL_CHOICE == 'logistic_regression':
    # Multi-class logistic regression using standard saga solver
    clf = LogisticRegression(solver='saga', multi_class='multinomial', max_iter=20, tol=0.1, random_state=42)
    print("Training Logistic Regression Model...")
    start_time = time.time()
    clf.fit(X_train_proc, y_train)
    print(f"Training complete in {time.time() - start_time:.2f} seconds!")

elif MODEL_CHOICE == 'knn':
    # k-NN classifier with k=5
    clf = KNeighborsClassifier(n_neighbors=5)
    print("Training k-NN Classifier (instantaneous lookup mapping)...")
    start_time = time.time()
    clf.fit(X_train_proc, y_train)
    print(f"Training complete in {time.time() - start_time:.2f} seconds!")

## Step 5: Evaluation
We generate an accuracy score, detailed classification report, and display the confusion matrix.

In [ ]:
# To prevent colab from hanging if k-NN is chosen, evaluate on a subset of the test set
if MODEL_CHOICE == 'knn':
    eval_size = 1000
    X_eval = X_test_proc[:eval_size]
    y_eval = y_test[:eval_size]
    print(f"Evaluating k-NN on a subset of {eval_size} test images to save time...")
else:
    X_eval = X_test_proc
    y_eval = y_test
    print(f"Evaluating Logistic Regression on all {len(y_test)} test images...")

# Predict
start_time = time.time()
y_pred = clf.predict(X_eval)
print(f"Inference complete in {time.time() - start_time:.2f} seconds!")

# Accuracy
accuracy = accuracy_score(y_eval, y_pred)
print(f"\nPipeline Accuracy: {accuracy * 100:.2f}%")

# Detailed Metrics
print("\nClassification Report:")
print(classification_report(y_eval, y_pred))

# Plot Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay.from_predictions(y_eval, y_pred, ax=ax, cmap='Blues')
plt.title(f'Confusion Matrix - {MODEL_CHOICE}')
plt.show()